# تشغيل موديل الـchurn على داتا جديدة

- النوتبوك ده **مش بيحفظ أي ملف** — بيعرض النتايج في الشاشة بس.
- الموديل: `models/prediction/churn_model.joblib` — الـthreshold ثابت 0.45.
- الداتا لازم تكون فيها الـ22 feature (نفس أعمدة `data/samples/sample_customers.csv`).
- شغّل الخلايا بالترتيب من فوق لتحت.

In [ ]:
from pathlib import Path

import joblib
import pandas as pd

# نلاقي جذر المشروع مهما كان مكان تشغيل النوتبوك
ROOT = Path.cwd()
while not (ROOT / "models" / "prediction" / "churn_model.joblib").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("مش لاقي جذر المشروع. شغّل النوتبوك من داخل المشروع.")
    ROOT = ROOT.parent

print("Project root:", ROOT)

In [ ]:
package = joblib.load(ROOT / "models" / "prediction" / "churn_model.joblib")

model = package["model"]
FEATURES = package["feature_columns"]
THRESHOLD = package["threshold"]
POS = list(model.classes_).index(package["positive_class"])

print(f"{len(FEATURES)} features | threshold = {THRESHOLD}")

## 1) اختار ملف الداتا

حط مسار ملفك في `INPUT_CSV`. لو سِبته فاضي، هيشتغل على ملف العينة.

المسار ممكن يكون كامل، أو نسبي من جذر المشروع.

In [ ]:
INPUT_CSV = ""   # مثال: r"E:\data\new_customers.csv"

path = Path(INPUT_CSV) if INPUT_CSV else ROOT / "data" / "samples" / "sample_customers.csv"
path = path if path.is_absolute() else ROOT / path

print("Reading:", path)
df = pd.read_csv(path)
print("rows:", len(df), "| columns:", len(df.columns))

## 2) فحص الأعمدة والقيم

In [ ]:
missing = [c for c in FEATURES if c not in df.columns]
if missing:
    raise ValueError(f"أعمدة ناقصة ({len(missing)}): {missing}")

X = df[FEATURES].apply(pd.to_numeric, errors="coerce")
bad = X.isna().sum()
bad = bad[bad > 0]
if len(bad):
    raise ValueError(f"في خلايا فاضية أو نص في الأعمدة دي:\n{bad}")

print("كل الأعمدة الـ22 سليمة ✔")

## 3) التوقع (احتمال + قرار)

In [ ]:
prob = model.predict_proba(X)[:, POS]

result = df.copy()
result["churn_probability"] = prob.round(4)
result["churn_prediction"] = (prob >= THRESHOLD).astype(int)

print(f"متوقع يسيب: {result['churn_prediction'].sum()} من {len(result)} عميل")
print(f"متوسط الاحتمال: {prob.mean():.3f}")

result[["churn_probability", "churn_prediction"]].head(10)

## 4) أعلى 10 عملاء في الخطورة

In [ ]:
cols = [c for c in ("Customer ID",) if c in result.columns]
cols += ["churn_probability", "churn_prediction"]

result.sort_values("churn_probability", ascending=False)[cols].head(10)

## ملاحظات

- مفيش أي ملف بيتحفظ من النوتبوك ده.
- `churn_prediction = 1` معناها الاحتمال ≥ 0.45.
- لو عايز تحفظ النتيجة في CSV، قولي وأنا أضيف الخلية (مش موجودة دلوقتي).